# Exploiting the subspace

Once $W_1$ is known, work in the active variables $y = W_1^\top x$ instead of all $m$
inputs. This tutorial uses the wing weight function, whose active subspace is
one-dimensional, to

- build the **active domain** (the set of possible $y$) and map between $x$ and $y$,
- fit a **surrogate** $g(y) \approx f(x)$,
- **integrate** $f$ (estimate its mean) with standard errors,
- **minimize** $f$ over the active variables and lift the minimizer back to $x$.

See {doc}`../user_guide/geometry`, {doc}`../user_guide/surrogates`,
{doc}`../user_guide/integration` and {doc}`../user_guide/optimization`, and for the
mathematics {doc}`../theory/geometry`, {doc}`../theory/surrogates_and_designs` and
{doc}`../theory/integration_and_optimization`.

In [ ]:
%matplotlib inline
import numpy as np

import active_subspaces as ac

wing = ac.test_functions.WING_WEIGHT
space = wing.space
X = space.sample(300, rng=1)
f, G = ac.evaluate_with_grads(wing, X=X)
sub = ac.active_subspace(grads=G, space=space, nboot=100, rng=2)
print("n =", sub.n)

## The active domain

For uniform inputs, $x$ ranges over the cube $[-1, 1]^m$, so $y = W_1^\top x$ ranges
over its image: an interval for $n = 1$ and a zonotope (a centrally symmetric polygon or
polytope) for $n \ge 2$. For Gaussian inputs the domain is all of $\mathbb R^n$.
`ac.active_domain` computes it exactly, with the cube corners that map to its vertices.

In [ ]:
dom = ac.active_domain(sub)
print(dom.kind, "with endpoints", dom.vertices.ravel())

`ac.forward_map` takes physical `X` to `(Y, Z)`, the active and inactive variables.
`ac.inverse_map` goes the other way: for a given $y$ it draws points $x$ with
$W_1^\top x = y$ from the input density restricted to that set (the "fiber"). Their
$f$ values vary little, because $f$ changes slowly in the inactive directions.

In [ ]:
Y, Z = ac.forward_map(sub, X=X)
inv = ac.inverse_map(sub, Y=np.array([[0.0]]), n_draws=8, rng=3)
print("8 points with y = 0, f =", np.round(ac.evaluate(wing, X=inv.X_physical), 1))
print(f"spread of f over the whole sample: {f.min():.1f} .. {f.max():.1f}")

## A surrogate on the active variable

`ac.fit_surrogate` fits a function of $y$ to the samples. It takes physical `X`; `kind`
is `"poly"` (least-squares polynomial) or `"rbf"` (radial basis functions with a
polynomial tail). `loo_r_squared` is a leave-one-out $R^2$, so it does not reward
overfitting.

In [ ]:
poly = ac.fit_surrogate(X=X, f=f, subspace=sub, kind="poly", degree=2)
rbf = ac.fit_surrogate(X=X, f=f, subspace=sub, kind="rbf")
print(f"leave-one-out R^2: quadratic {poly.loo_r_squared:.4f}, RBF {rbf.loo_r_squared:.4f}")
fig, ax = ac.plotting.plot_surrogate_fit(poly, X=X, f=f)
fig

`ac.predict` evaluates a surrogate at physical points; `poly.evaluate` takes active
coordinates `Y` directly.

In [ ]:
X_new = space.sample(5, rng=4)
print("surrogate:", np.round(ac.predict(poly, X_new), 1))
print("model:    ", np.round(ac.evaluate(wing, X=X_new), 1))

## Integration

The mean of $f$ splits into an integral over $y$ and an average over the inactive
variables $z$:

$$\mathbb E[f] = \int G(y)\,\rho_Y(y)\,dy, \qquad G(y) = \mathbb E[f \mid y].$$

`ac.integrate` uses a quadrature rule with `n_nodes` nodes in $y$ (a Gauss rule on an
interval) and estimates each $G(y_k)$ by the mean of $f$ at `n_mc` points drawn from the
fiber (default 50). It therefore costs `n_nodes * n_mc` model evaluations. The
`inactive_std_error` measures the Monte Carlo error of the fiber averages only; the
quadrature error in $y$ is not included.

In [ ]:
res = ac.integrate(wing, domain=dom, n_nodes=5, rng=5)
print(f"estimate {res.estimate:.2f} +/- {res.inactive_std_error:.2f}")
print(f"model evaluations: {res.f.size}")

With `n_replicates` of at least 2 the whole estimate is repeated with independent
draws, and `total_std_error` is the standard error of their mean. Compare plain Monte
Carlo on the 300 samples, and the reference mean from $10^6$ samples:

In [ ]:
rep = ac.integrate(wing, domain=dom, n_nodes=5, n_replicates=10, rng=6)
print(f"10 replicates: {rep.estimate:.2f} +/- {rep.total_std_error:.2f}, {rep.f.size} evaluations")
print(f"plain Monte Carlo, 300 samples: {f.mean():.2f} +/- {f.std(ddof=1) / np.sqrt(f.size):.2f}")
print(f"reference: {wing.reference_results.mean:.2f} +/- {wing.reference_results.mean_se:.2f}")

Integrating the surrogate instead costs no model evaluations; its error is the
surrogate's error, which the result cannot estimate (`inactive_std_error` is `None`).

In [ ]:
res_s = ac.integrate_surrogate(poly, domain=dom, n_nodes=5)
print(f"surrogate mean: {res_s.estimate:.2f}  (standard error: {res_s.inactive_std_error})")

## Optimization

`ac.minimize_active` minimizes the surrogate $g(y)$ over the active domain. `ac.minimize`
does the same and then **lifts** $y^\ast$ to a full input $x^\ast$ with
$W_1^\top x^\ast = y^\ast$: it fits a quadratic model of $f$ to all the samples (in the
coordinates of the first few eigenvectors) and chooses the inactive variables $z$ to
minimize it with $y = y^\ast$ fixed, within the bounds. With `model=`, it also
evaluates $f(x^\ast)$.

**The lift in v1.** The lift is a heuristic without a published reference. It
minimizes the quadratic in $z$ with its curvature matrix $Q = V \operatorname{diag}(\theta_i)
V^\top$ replaced by $V \operatorname{diag}(\max(\theta_i, \tau)) V^\top$, with
$\tau = 10^{-2}\max_i\lvert\theta_i\rvert$ (parameter `curvature_floor`), so the problem
stays convex. The 2016 code added $0.01 I$ instead. When any eigenvalue is raised, the
library warns with `AccuracyWarning`, code `accuracy.curvature_clipped`, as below: the
wing weight function is nearly flat in some inactive directions.

In [ ]:
opt_y = ac.minimize_active(poly, domain=dom, rng=7)
print(f"y* = {opt_y.y_star[0]:.3f}, g(y*) = {opt_y.value:.1f}")

opt = ac.minimize(poly, domain=dom, X=X, f=f, model=wing, rng=8)
print(f"g(y*) = {opt.f_star:.1f}, f(x*) = {opt.f_model:.1f}, smallest sampled f = {f.min():.1f}")
print(
    "x* =",
    "  ".join(f"{k}={v:.4g}" for k, v in zip(space.names, opt.x_star_physical, strict=True)),
)

$y^\ast$ lies at an end of the interval, so $x^\ast$ sits at a corner of the box,
and the model value there is far below anything in the sample. The surrogate value
$g(y^\ast)$ is an extrapolation of the quadratic and differs from $f(x^\ast)$; trust
the model value.

## Two active variables

With $n = 2$ the active domain of a uniform space is a zonotope. `ac.active_design`
places design points in it (a maximin design), and `plot_active_domain` draws the
domain, the design and the samples coloured by $f$.

In [ ]:
sub2 = ac.with_dimension(sub, 2)
dom2 = ac.active_domain(sub2)
design = ac.active_design(domain=dom2, n_points=15, rng=9)
print(dom2.kind, "with", len(dom2.vertices), "vertices")
fig, ax = ac.plotting.plot_active_domain(
    dom2, design=design.points, points=ac.forward_map(sub2, X=X).Y, values=f
)
fig

Integration works the same way. On a zonotope the rule in $y$ is built from a random
maximin design and Monte Carlo cell masses, and its error is not part of
`inactive_std_error`. With `n_replicates`, the rule is redrawn for each replicate, so
`total_std_error` accounts for it as well.

In [ ]:
res2 = ac.integrate(wing, domain=dom2, n_nodes=10, n_replicates=5, rng=10)
print(
    f"n = 2: estimate {res2.estimate:.2f}; inactive std error {res2.inactive_std_error:.2f}, "
    f"total std error {res2.total_std_error:.2f}"
)

{doc}`reduced_model` packages these steps behind a single object.

## References

- Constantine, P. G. (2015). *Active Subspaces: Emerging Ideas for Dimension Reduction
  in Parameter Studies.* SIAM.